# BENCH v4: Structural Equation Model Validation
## Testing the BENCH behavioural framework with Niamir et al. (2020) survey data

**Objective**: Use the ES household energy survey (Niamir et al. 2020, *ERSS*) to test whether the causal structure assumed in the BENCH agent-based model is supported by empirical data.

**Reference model**: Li et al. (2025) extended TPB–SEM (*Energy & Buildings*) — adapted to the BENCH behavioural framework (Niamir et al. 2020, *Climatic Change*).

**BENCH decision chain**:
```
Knowledge Activation (CEEK, CEEA, EDA)
    → Guilt / Moral obligation
    → Personal Norms (PN)  ←→  Social Norms (SN)
    → Motivation (PN + SN)
    → Consideration (PBC1 investment | PBC2 conservation | PBC3 switching)
    → Renovation action (I1-I3 | C1-C3 | S1-S3)
```

**Survey dataset** (gitignored — not committed to repo):
- ES: `survey_population/HHSurvey_BENCH_LNiamir_ES_DB.csv` (n = 755, raw item-level data)

---
**Analysis plan**:
1. Data loading and variable mapping
2. Cross-loading analysis (items shared across constructs)
3. Descriptive statistics and Spearman correlations
4. Reliability analysis (Cronbach's α)
5. SEM specification and strategy
6. Model fitting and fit indices (CFI, RMSEA, SRMR)
7. Path diagram

## 0. Setup

In [17]:

# Install SEM and supporting libraries via uv (project package manager)
import subprocess, sys
subprocess.run(
    ['uv', 'pip', 'install', 'semopy', 'factor_analyzer', 'pingouin'],
    check=False
)


CompletedProcess(args=['uv', 'pip', 'install', 'semopy', 'factor_analyzer', 'pingouin'], returncode=0)

In [18]:
import os
import sys
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import seaborn as sns

warnings.filterwarnings('ignore')

# Add project root to path AND set cwd so relative file paths in
# build_population / question_key resolve correctly regardless of where
# Jupyter was launched from.
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from survey_population.build_population import load_real_population, _load_excel, build
from survey_population.question_key import (
    VARIABLE_QUESTIONS, DATA_FILES, SHEET_NAME, SURVEY_YEAR
)

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb',
    'axes.facecolor':   '#fcfcfb',
    'axes.spines.top':  False,
    'axes.spines.right': False,
    'axes.spines.left': True,
    'axes.spines.bottom': True,
    'font.family': 'sans-serif',
})

C_ES = '#2a78d6'   # blue — Spain
C_NL = '#e34948'   # red  — Netherlands
C_GRAY = '#8a8a8a'

BEH_VARS = ['ceek', 'ceea', 'eda', 'pn', 'sn', 'pbc1', 'pbc2', 'pbc3']

# Binary behavioral outcome items (Q8xx present in raw ES survey)
# Q801–Q807: all 755 respondents answered; 0 = no, 1 = yes in past 10 years
# NOTE: exact action mapping needs to be verified against the questionnaire
BEHAVIOR_ITEMS = ['Q801', 'Q802', 'Q803', 'Q804', 'Q805', 'Q806', 'Q807']

print('Setup complete.  Working directory:', os.getcwd())

Setup complete.  Working directory: C:\Users\torrenperaire\Documents


## 1. Data Loading

In [19]:
# --- Processed composites (19 BENCH variables) ---
df_es = load_real_population('ES')   # raw survey → build() → 19 composites

print(f'ES processed respondents : {len(df_es):,} × {df_es.shape[1]} variables')

FileNotFoundError: Survey data for ES not found: survey_population/HHSurvey_BENCH_LNiamir_ES_DB.csv
Place the file at that path before running.

In [ ]:
# --- Raw ES survey (item-level access for reliability analysis + behavior outcomes) ---
raw_es = _load_excel(DATA_FILES['ES'], SHEET_NAME)
print(f'ES raw survey : {raw_es.shape[0]:,} rows × {raw_es.shape[1]} columns')

# Convert behavior items to numeric and attach to processed ES data
behav_raw = raw_es[BEHAVIOR_ITEMS].apply(pd.to_numeric, errors='coerce')

# The processed df_es has the same row order as raw_es (build() only drops all-NaN rows;
# check alignment via Respondent_ID if present)
if 'Respondent_ID' in raw_es.columns:
    # Ensure we align on row index — build() preserves order so direct concat works
    pass

# Simple concat (same order guaranteed by build_population.build)
df_es_full = pd.concat(
    [df_es.reset_index(drop=True), behav_raw.reset_index(drop=True)],
    axis=1
)

print(f'\nCombined ES dataframe: {df_es_full.shape[0]:,} rows × {df_es_full.shape[1]} columns')
df_es_full[BEH_VARS + BEHAVIOR_ITEMS].head(3)

ES raw survey : 755 rows × 509 columns

Combined ES dataframe: 755 rows × 26 columns


,ceek,ceea,eda,pn,sn,pbc1,pbc2,pbc3,Q801,Q802,Q803,Q804,Q805,Q806,Q807
0,4.5,4.428571,5.333333,5.250,5.600000,4.6,3.5,4.5,1,0,1,0,0,0,0
1,3.5,4.000000,4.000000,3.750,3.400000,3.6,3.5,4.0,1,0,0,0,0,1,0
2,3.0,2.857143,3.000000,3.125,2.833333,3.2,4.5,4.5,0,0,0,1,0,0,0


In [ ]:
# Behavioral outcome prevalence
behav_summary = pd.DataFrame({
    'Item': BEHAVIOR_ITEMS,
    'N (yes)': [df_es_full[c].sum() for c in BEHAVIOR_ITEMS],
    '% yes': [(df_es_full[c].mean() * 100).round(1) for c in BEHAVIOR_ITEMS],
    'Label (needs verification)': [
        'Insulation / building envelope',
        'Windows / glazing upgrade',
        'Heating system investment',
        'Renewable energy / solar',
        'Fuel switching 1',
        'Fuel switching 2',
        'Conservation / behaviour change',
    ]
})
behav_summary

,Item,N (yes),% yes,Label (needs verification)
0,Q801,122,16.2,Insulation / building envelope
1,Q802,93,12.3,Windows / glazing upgrade
2,Q803,155,20.5,Heating system investment
3,Q804,76,10.1,Renewable energy / solar
4,Q805,70,9.3,Fuel switching 1
5,Q806,78,10.3,Fuel switching 2
6,Q807,413,54.7,Conservation / behaviour change


## 2. Variable Mapping and Item Overlap

Each BENCH construct is computed as the **mean of its contributing survey items**. Before fitting a CFA or SEM at the item level, we need to check which items appear in more than one construct — these **cross-loaded items** would violate the simple structure assumed by standard CFA.


In [ ]:
# Variable mapping table
rows = []
for var, items in VARIABLE_QUESTIONS.items():
    rows.append({
        'Construct': var.upper(),
        'Role in BENCH': {
            'ceek': 'Knowledge (climate-energy-economy)',
            'ceea': 'Awareness (climate-energy-economy)',
            'eda':  'Energy decision awareness',
            'pn':   'Personal moral norms (motivation)',
            'sn':   'Social norms (motivation)',
            'pbc1': 'PBC — investment feasibility',
            'pbc2': 'PBC — conservation feasibility',
            'pbc3': 'PBC — fuel-switching feasibility',
        }[var],
        'N items': len(items),
        'Items': ', '.join(items),
    })

mapping_df = pd.DataFrame(rows).set_index('Construct')
mapping_df

,Role in BENCH,N items,Items
Construct,,,
CEEK,Knowledge (climate-energy-economy),6,"Q70_1, Q70_2, Q70_3, Q100_1, Q100_2, Q70_7"
CEEA,Awareness (climate-energy-economy),14,"Q60_1, Q60_2, Q60_3, Q60_4, Q60_5, Q60_6, Q60_..."
EDA,Energy decision awareness,5,"Q70_4, Q70_6, Q70_8, Q100_4, Q100_6"
PN,Personal moral norms (motivation),8,"Q70_6, Q70_7, Q70_8, Q100_2, Q100_3, Q100_4, Q..."
SN,Social norms (motivation),6,"Q100_5, Q540_4, Q540_6, Q540_7, Q540_8, Q540_10"
PBC1,PBC — investment feasibility,5,"Q450_2, Q450_4, Q450_6, Q540_3, Q540_5"
PBC2,PBC — conservation feasibility,2,"Q540_1, Q540_2"
PBC3,PBC — fuel-switching feasibility,2,"Q450_1, Q450_5"


In [ ]:
# Identify cross-loaded items
item_to_constructs: dict[str, list] = defaultdict(list)
for var, items in VARIABLE_QUESTIONS.items():
    for item in items:
        item_to_constructs[item].append(var.upper())

cross_loaded = {item: cs for item, cs in sorted(item_to_constructs.items()) if len(cs) > 1}
unique_items  = {item for item, cs in item_to_constructs.items() if len(cs) == 1}

print(f'Total unique question items : {len(item_to_constructs)}')
print(f'Cross-loaded (in ≥2 constructs): {len(cross_loaded)} ({100*len(cross_loaded)/len(item_to_constructs):.0f}%)')
print(f'Unique (in exactly 1 construct): {len(unique_items)}\n')

print('Cross-loaded items:')
for item, cs in cross_loaded.items():
    print(f'  {item:<10}  →  {" + ".join(cs)}')

Total unique question items : 37
Cross-loaded (in ≥2 constructs): 10 (27%)
Unique (in exactly 1 construct): 27

Cross-loaded items:
  Q100_2      →  CEEK + PN
  Q100_4      →  EDA + PN
  Q100_6      →  EDA + PN
  Q70_1       →  CEEK + CEEA
  Q70_2       →  CEEK + CEEA
  Q70_3       →  CEEK + CEEA
  Q70_4       →  CEEA + EDA
  Q70_6       →  EDA + PN
  Q70_7       →  CEEK + CEEA + PN
  Q70_8       →  EDA + PN


### Item overlap — implications for SEM

Roughly **half the items are shared across constructs**. For standard CFA this is a major problem: items are expected to load on exactly one latent factor. Cross-loaded items produce model misfit and biased factor loadings.

**Three strategies**:

| Strategy | When to use | Trade-off |
|---|---|---|
| **A. Composite-level path analysis** | Our main choice | No item-level measurement model; treats composites as observed; no cross-loading issue |
| **B. ESEM / bifactor CFA** | If we need item-level loadings | Allows cross-loadings; more complex to interpret |
| **C. Unique-item CFA** | If cross-loaded items are reassigned to one factor | Reduces overlap but changes construct definitions |

**Decision**: We use **Strategy A** as the primary analysis — the 8 composite scores (ceek, ceea, eda, pn, sn, pbc1, pbc2, pbc3) are treated as observed indicators in a **path model** / **hybrid SEM**. We will use item-level data only for reliability analysis (Cronbach's α) in Section 4.


## 3. Descriptive Statistics

In [ ]:
# Summary statistics — ES
stats_es = df_es[BEH_VARS].describe().T[['count', 'mean', 'std', 'min', 'max']].round(3)
stats_es.index.name = 'Construct'
stats_es

In [ ]:
# Distribution — ES violin plots
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Distribution of BENCH behavioural composites — ES (1–7 Likert)', fontsize=13, y=1.01)

for ax, var in zip(axes.flatten(), BEH_VARS):
    data_es = df_es[var].dropna().values
    parts = ax.violinplot([data_es], positions=[0], widths=0.7, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor(C_ES)
        pc.set_alpha(0.6)
    for el in ['cmedians', 'cbars', 'cmins', 'cmaxes']:
        if el in parts:
            parts[el].set_color(C_ES)
    ax.set_title(var.upper(), fontsize=10)
    ax.set_xticks([0])
    ax.set_xticklabels([f'ES (n={len(df_es):,})'], fontsize=8)
    ax.set_ylim(0.5, 7.5)
    ax.set_yticks([1, 2, 3, 4, 5, 6, 7])
    ax.grid(axis='y', alpha=0.3, linewidth=0.5)

plt.tight_layout()
plt.savefig('distributions_es.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation Analysis

Spearman correlations between constructs. The BENCH causal structure predicts:
- **Knowledge → PN/SN**: CEEK, CEEA, EDA should correlate positively with PN and SN
- **PN/SN → PBC**: Norms should correlate positively with PBC feasibility measures
- **Intra-knowledge**: CEEK ↔ CEEA ↔ EDA should correlate strongly (shared domain)


In [ ]:
# Spearman correlation matrix — ES
corr_es = df_es[BEH_VARS].corr(method='spearman')

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.zeros_like(corr_es, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(
    corr_es, mask=mask,
    annot=True, fmt='.2f', annot_kws={'size': 10},
    cmap='RdBu_r', center=0, vmin=-0.6, vmax=0.6,
    ax=ax,
    linewidths=0.5, linecolor='#e1e0d9',
    cbar_kws={'shrink': 0.75, 'label': 'Spearman r'},
    square=True,
)
ax.set_title(f'ES — Spain (Navarre)  (n = {len(df_es):,})', fontsize=12, pad=10)
ax.set_xticklabels([v.upper() for v in BEH_VARS], rotation=45, ha='right', fontsize=9)
ax.set_yticklabels([v.upper() for v in BEH_VARS], rotation=0, fontsize=9)

plt.tight_layout()
plt.savefig('correlation_matrix_es.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Print key predicted correlations
PREDICTED_PATHS = [
    ('ceek', 'ceea', 'CEEK ↔ CEEA  (intra-knowledge)'),
    ('ceek', 'eda',  'CEEK ↔ EDA   (intra-knowledge)'),
    ('ceea', 'eda',  'CEEA ↔ EDA   (intra-knowledge)'),
    ('ceek', 'pn',   'CEEK → PN    (knowledge → personal norm)'),
    ('ceea', 'pn',   'CEEA → PN    (awareness → personal norm)'),
    ('eda',  'pn',   'EDA  → PN    (decision awareness → personal norm)'),
    ('ceek', 'sn',   'CEEK → SN    (knowledge → social norm)'),
    ('ceea', 'sn',   'CEEA → SN    (awareness → social norm)'),
    ('pn',   'sn',   'PN   ↔ SN    (between motivation constructs)'),
    ('pn',   'pbc1', 'PN   → PBC1  (norms → investment feasibility)'),
    ('pn',   'pbc2', 'PN   → PBC2  (norms → conservation feasibility)'),
    ('pn',   'pbc3', 'PN   → PBC3  (norms → switching feasibility)'),
    ('sn',   'pbc1', 'SN   → PBC1  (social norm → investment)'),
    ('sn',   'pbc2', 'SN   → PBC2  (social norm → conservation)'),
    ('sn',   'pbc3', 'SN   → PBC3  (social norm → switching)'),
]

print(f'{"Path":<50}  {"ES r":>7}')
print('-' * 62)
for a, b, label in PREDICTED_PATHS:
    r_es = corr_es.loc[a, b]
    print(f'  {label:<48}  {r_es:>7.3f}')

## 5. Reliability Analysis (Cronbach's α)

Computed at the **item level** using the raw ES survey. NL is pre-processed so item-level α cannot be computed for NL.

Thresholds (conventional): α ≥ 0.70 acceptable, ≥ 0.80 good, ≥ 0.90 excellent.

> **Note on cross-loaded items**: Items shared across constructs contribute to α for each construct they appear in, which can inflate α estimates. These are noted below.


In [ ]:
def cronbach_alpha(df_items: pd.DataFrame) -> float:
    """Cronbach's α from a DataFrame of item columns."""
    data = df_items.apply(pd.to_numeric, errors='coerce').dropna()
    n = data.shape[1]
    if n < 2 or len(data) < 5:
        return np.nan
    total_var = data.sum(axis=1).var(ddof=1)
    item_vars = data.var(axis=0, ddof=1).sum()
    if total_var == 0:
        return np.nan
    return (n / (n - 1)) * (1 - item_vars / total_var)


def item_total_correlations(df_items: pd.DataFrame) -> pd.Series:
    """Pearson correlation of each item with the sum of all other items."""
    data = df_items.apply(pd.to_numeric, errors='coerce').dropna()
    total = data.sum(axis=1)
    results = {}
    for col in data.columns:
        rest = total - data[col]
        results[col] = data[col].corr(rest)
    return pd.Series(results)


# Compute α and item-total correlations for each construct
rows = []
for var, items in VARIABLE_QUESTIONS.items():
    avail = [c for c in items if c in raw_es.columns]
    if not avail:
        continue
    sub = raw_es[avail]
    α = cronbach_alpha(sub)
    itc = item_total_correlations(sub)
    cross_items = [c for c in avail if len(item_to_constructs[c]) > 1]
    rows.append({
        'Construct': var.upper(),
        'N items': len(avail),
        'N (complete)': sub.apply(pd.to_numeric, errors='coerce').dropna().shape[0],
        "Cronbach's α": round(α, 3),
        'Quality': 'Excellent' if α >= 0.90 else ('Good' if α >= 0.80 else ('Acceptable' if α >= 0.70 else 'Low')),
        'Cross-loaded items': ', '.join(cross_items) if cross_items else '—',
        'Min item-total r': round(itc.min(), 3),
    })

reliability_df = pd.DataFrame(rows).set_index('Construct')
reliability_df

In [ ]:
# Item-total correlations detail — check for low-performing items
print('Item-total correlations (should be > 0.30 for item retention):\n')
for var, items in VARIABLE_QUESTIONS.items():
    avail = [c for c in items if c in raw_es.columns]
    if not avail:
        continue
    itc = item_total_correlations(raw_es[avail])
    low = itc[itc < 0.30]
    print(f'  {var.upper():<6}  mean r = {itc.mean():.3f}', end='')
    if len(low) > 0:
        print(f'   ⚠  Low items: {dict(low.round(3))}')
    else:
        print('   ✓')

## 6. Behavior Outcome Summary (ES only)

Q801–Q807 are the energy renovation/behaviour outcomes. We will use these as endogenous variables in the structural model. Since they are binary (0/1), we create:
- **`invest_score`**: mean of Q801–Q804 (investment-type, lower uptake ≈ 10–20%)
- **`switch_score`**: mean of Q805–Q806 (fuel switching, ≈ 10%)
- **`conserve_score`**: Q807 (conservation behaviour, ≈ 55%)

> **⚠ To do**: verify Q80x → I/C/S mapping against the questionnaire before interpreting path coefficients.


In [ ]:
# Create composite behavior scores (0–1 continuous)
df_es_full['invest_score']   = df_es_full[['Q801', 'Q802', 'Q803', 'Q804']].mean(axis=1)
df_es_full['switch_score']   = df_es_full[['Q805', 'Q806']].mean(axis=1)
df_es_full['conserve_score'] = df_es_full['Q807'].astype(float)

# Also create a total renovation score
df_es_full['total_renovations'] = df_es_full[BEHAVIOR_ITEMS].sum(axis=1)

OUTCOME_VARS = ['invest_score', 'switch_score', 'conserve_score', 'total_renovations']

print('Behavioral outcome summaries:')
print(df_es_full[OUTCOME_VARS].describe().T[['count', 'mean', 'std', 'min', 'max']].round(3))

In [ ]:
# Correlation of behavioral outcomes with predictors
outcome_corr = df_es_full[BEH_VARS + OUTCOME_VARS].corr(method='spearman')
predictor_outcome_corr = outcome_corr.loc[BEH_VARS, OUTCOME_VARS]

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    predictor_outcome_corr,
    annot=True, fmt='.2f', annot_kws={'size': 10},
    cmap='RdBu_r', center=0, vmin=-0.4, vmax=0.4,
    ax=ax, linewidths=0.5, linecolor='#e1e0d9',
    cbar_kws={'shrink': 0.8, 'label': 'Spearman r'},
    square=True,
)
ax.set_title('Spearman correlations: BENCH predictors → behavioral outcomes (ES)', fontsize=11)
ax.set_xticklabels(['Investment', 'Switching', 'Conservation', 'Total'], rotation=25, ha='right', fontsize=9)
ax.set_yticklabels([v.upper() for v in BEH_VARS], rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('predictor_outcome_correlations.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. SEM Strategy and Model Specification

### 7.1 Why composite-level path analysis?

Given the item cross-loading (Section 2), we treat the **8 BENCH composites as observed variables** rather than fitting item-level latent factors. This approach:
- Avoids the cross-loading problem
- Preserves the BENCH construct definitions exactly
- Is equivalent to what Niamir et al. (2020) already validated when building the composites

### 7.2 BENCH behavioural pathway as SEM

The BENCH causal chain maps directly onto a path model:

```
# BENCH chain (Knowledge → Norms → PBC)
pn ~ ceek + ceea + eda          # Knowledge activates moral guilt → personal norm
sn ~ ceek + ceea + eda          # Knowledge activates social awareness → social norm
pbc1 ~ pn + sn                  # Motivation → investment consideration
pbc2 ~ pn + sn                  # Motivation → conservation consideration
pbc3 ~ pn + sn                  # Motivation → switching consideration

# Theoretically-justified residual correlations
pn ~~ sn                        # PN and SN are mutually reinforcing (both = Motivation)
pbc1 ~~ pbc2                    # General household capacity factor
pbc1 ~~ pbc3                    #   (income/access affects all PBC measures)
pbc2 ~~ pbc3                    #   beyond the norms-explained variance
```

The `pn ~~ sn` correlation is supported by the BENCH model itself: both are components of "Motivation" — a household that feels strong moral obligation (PN) also tends to perceive stronger social norms (SN). The PBC residual correlations represent a general household-level facilitation factor (resources, information access) not captured by norms alone.

### 7.3 Model variants

| Model | Specification | CFI (ES) | RMSEA (ES) |
|---|---|---|---|
| **M1: Base path** | ceek/ceea/eda → pn/sn → pbc | 0.915 | 0.154 |
| **M1b: Primary ✓** | M1 + PN↔SN + PBC residuals | **0.993** | **0.049** |
| **M1c: Direct K→PBC** | M1b + ceek/ceea/eda → pbc directly | 0.942 | 0.184 |
| **M2: Hybrid SEM** | Latent Knowledge → pn/sn → pbc | 0.865 | 0.205 |

**Key finding from model comparison**: M1b fits excellently. M1c (adding direct knowledge → PBC paths) *worsens* fit — confirming that knowledge reaches PBC **only through norms**, exactly as BENCH assumes. The latent Knowledge factor (M2) performs poorly because CEEK, CEEA, and EDA have different loadings on PN and SN (they're not interchangeable indicators of a single construct).

### 7.4 Comparison with Li et al. (2025)

| Li et al. (2025) construct | BENCH equivalent | Source in survey |
|---|---|---|
| Attitudes (ATT) | Personal Norms (PN) | Q70_6-8, Q100_2-4, Q540_9 |
| Normative beliefs (NM) | Social Norms (SN) | Q100_5, Q540_4,6-8,10 |
| PBC | PBC1 + PBC2 + PBC3 | Q450_*, Q540_1-5 |
| Antecedents (PB, WH, CC) | Knowledge (CEEK, CEEA, EDA) | Q60_*, Q70_*, Q100_1-2 |
| Intention (INT) | **Not measured** | — |
| Energy saving behaviour (ESB) | Q801–Q807 binary outcomes | Q8xx |

Key difference: BENCH has **no separate intention measure** in the survey. Intention emerges from the motivation × consideration calculation in the ABM, so we test the path directly (Norms + PBC → Behavior).


## 8. Model Fitting

We use `semopy` (version ≥ 2.3) which implements ML estimation for structural equation models with lavaan-style syntax.

**Fit index benchmarks** (Hu & Bentler 1999):
- CFI ≥ 0.95 (good), ≥ 0.90 (acceptable)
- RMSEA ≤ 0.06 (good), ≤ 0.08 (acceptable)
- SRMR ≤ 0.08 (good)
- χ²/df ≤ 3.0 (acceptable)


In [ ]:
import semopy
print(f'semopy version: {semopy.__version__}')

In [ ]:

# ── M1: Base path model ───────────────────────────────────────────────────────
M1_SPEC = """
pn ~ ceek + ceea + eda
sn ~ ceek + ceea + eda
pbc1 ~ pn + sn
pbc2 ~ pn + sn
pbc3 ~ pn + sn
"""

# ── M1b: Primary model — adds theoretically justified residual correlations ───
M1B_SPEC = """
pn ~ ceek + ceea + eda
sn ~ ceek + ceea + eda
pbc1 ~ pn + sn
pbc2 ~ pn + sn
pbc3 ~ pn + sn
pn ~~ sn
pbc1 ~~ pbc2
pbc1 ~~ pbc3
pbc2 ~~ pbc3
"""

# ── M1c: Alternative — direct knowledge → PBC paths (should worsen fit) ──────
M1C_SPEC = """
pn ~ ceek + ceea + eda
sn ~ ceek + ceea + eda
pbc1 ~ pn + sn + ceek + ceea + eda
pbc2 ~ pn + sn + ceek + ceea + eda
pbc3 ~ pn + sn + ceek + ceea + eda
pn ~~ sn
"""

# ── M2: Hybrid SEM with latent Knowledge ─────────────────────────────────────
M2_SPEC = """
Knowledge =~ ceek + ceea + eda
pn ~ Knowledge
sn ~ Knowledge
pbc1 ~ pn + sn
pbc2 ~ pn + sn
pbc3 ~ pn + sn
pn ~~ sn
pbc1 ~~ pbc2
pbc1 ~~ pbc3
pbc2 ~~ pbc3
"""

data_es = df_es[BEH_VARS].dropna()

print('Fitting 4 models...')
models = {}
for spec_name, spec in [('M1', M1_SPEC), ('M1b', M1B_SPEC), ('M1c', M1C_SPEC), ('M2', M2_SPEC)]:
    key = f'{spec_name}_ES'
    m = semopy.Model(spec)
    m.fit(data_es)
    models[key] = m
    s = semopy.calc_stats(m).iloc[0]
    print(f'  {key:<10}  CFI={s["CFI"]:.3f}  RMSEA={s["RMSEA"]:.3f}  df={s["DoF"]:.0f}')


In [ ]:

# Fit statistics table
rows = []
for spec_name in ['M1', 'M1b', 'M1c', 'M2']:
    m = models[f'{spec_name}_ES']
    s = semopy.calc_stats(m).iloc[0]
    rows.append({
        'Model': spec_name,
        'df': int(s['DoF']),
        'χ²': round(s['chi2'], 1),
        'CFI': round(s['CFI'], 3),
        'RMSEA': round(s['RMSEA'], 3),
        'AIC': round(s['AIC'], 1),
    })

fit_table = pd.DataFrame(rows)
fit_table['Fit'] = fit_table.apply(
    lambda r: '✓ Good' if r['CFI'] >= 0.95 and r['RMSEA'] <= 0.06
    else ('~ Accept' if r['CFI'] >= 0.90 and r['RMSEA'] <= 0.08 else '✗ Poor'),
    axis=1
)
fit_table.set_index('Model')


In [ ]:

# ── M3: Full model — adds behavioral outcomes (ES only) ──────────────────────
M3_SPEC = """
pn ~ ceek + ceea + eda
sn ~ ceek + ceea + eda
pbc1 ~ pn + sn
pbc2 ~ pn + sn
pbc3 ~ pn + sn
pn ~~ sn
pbc1 ~~ pbc2
pbc1 ~~ pbc3
pbc2 ~~ pbc3

invest_score   ~ pbc1 + pn + sn
switch_score   ~ pbc3 + pn + sn
conserve_score ~ pbc2 + pn + sn
"""

m3_data = df_es_full[BEH_VARS + ['invest_score', 'switch_score', 'conserve_score']].dropna()
m3_es = semopy.Model(M3_SPEC)
m3_es.fit(m3_data)

s3 = semopy.calc_stats(m3_es).iloc[0]
print(f'M3 (full, ES):  CFI={s3["CFI"]:.3f}  RMSEA={s3["RMSEA"]:.3f}  df={s3["DoF"]:.0f}')


In [ ]:

# Complete fit table including M3
s3 = semopy.calc_stats(m3_es).iloc[0]
m3_row = pd.DataFrame([{
    'Model': 'M3', 'df': int(s3['DoF']),
    'χ²': round(s3['chi2'], 1), 'CFI': round(s3['CFI'], 3),
    'RMSEA': round(s3['RMSEA'], 3), 'AIC': round(s3['AIC'], 1),
    'Fit': '✓ Good' if s3['CFI'] >= 0.95 and s3['RMSEA'] <= 0.06 else '~ Accept',
}])

full_fit_table = pd.concat([fit_table, m3_row], ignore_index=True)
full_fit_table.set_index('Model')


## 9. Path Coefficients


In [ ]:

# ── Module-level helpers for extracting path estimates ────────────────────────

def get_structural_paths(model):
    """Return a DataFrame of regression paths (~) from a fitted semopy model."""
    try:
        p = model.inspect(mode='list', what='est', information='expected')
    except Exception:
        p = model.inspect()
    if 'op' in p.columns:
        p = p[p['op'] == '~']
    return p


def lookup_est(df_paths, lhs, rhs):
    """Look up the estimated coefficient for a single path lhs ~ rhs."""
    try:
        row = df_paths[(df_paths['lval'] == lhs) & (df_paths['rval'] == rhs)]
        return float(row['Estimate'].iloc[0]) if len(row) > 0 else np.nan
    except Exception:
        return np.nan


def get_params(model, label):
    """Return all parameter estimates with a Dataset column."""
    try:
        params = model.inspect(mode='list', what='est', information='expected')
    except Exception:
        params = model.inspect()
    params = params.copy()
    params['Dataset'] = label
    return params


# ── Primary model is M1b ──────────────────────────────────────────────────────
m1b_es = models['M1b_ES']

params_es = get_params(m1b_es, 'ES')
reg_es = params_es[params_es['op'] == '~'][['lval', 'rval', 'Estimate', 'Std. Err', 'z-value', 'p-value']]

print('=== M1b Path estimates — ES ===')
display(reg_es.round(4))


In [ ]:

# R² for endogenous variables in M1b
def get_r2(model):
    try:
        return model.inspect(what='r2')
    except Exception:
        return None

r2_es = get_r2(m1b_es)
print('R² — M1b (ES):', r2_es)


## 10. Path Diagram Visualization


In [ ]:

def draw_bench_sem_path_diagram(model_es, title='BENCH SEM — M1b (primary model)'):
    """
    Path diagram for M1b: ceek/ceea/eda → pn/sn → pbc1/2/3.
    Unstandardized structural coefficients for ES.
    Residual correlations drawn as curved double-headed arrows.
    """
    p_es = get_structural_paths(model_es)

    fig, ax = plt.subplots(figsize=(13, 8))
    ax.set_xlim(0, 11)
    ax.set_ylim(0, 8)
    ax.axis('off')
    fig.patch.set_facecolor('#fcfcfb')
    ax.set_facecolor('#fcfcfb')

    pos = {
        'ceek': (1.3, 6.5), 'ceea': (1.3, 4.5), 'eda': (1.3, 2.5),
        'pn':   (4.5, 6.0), 'sn':   (4.5, 3.0),
        'pbc1': (8.0, 6.5), 'pbc2': (8.0, 4.5), 'pbc3': (8.0, 2.5),
    }
    labels = {
        'ceek': 'CEEK', 'ceea': 'CEEA', 'eda': 'EDA',
        'pn': 'Personal\nNorms (PN)', 'sn': 'Social\nNorms (SN)',
        'pbc1': 'PBC1\n(Investment)', 'pbc2': 'PBC2\n(Conservation)', 'pbc3': 'PBC3\n(Switching)',
    }

    node_base = {'boxstyle': 'round,pad=0.5', 'linewidth': 1.5, 'zorder': 5}
    node_styles = {
        'ceek': {**node_base, 'facecolor': '#e6f2fb', 'edgecolor': '#2a78d6'},
        'ceea': {**node_base, 'facecolor': '#e6f2fb', 'edgecolor': '#2a78d6'},
        'eda':  {**node_base, 'facecolor': '#e6f2fb', 'edgecolor': '#2a78d6'},
        'pn':   {**node_base, 'facecolor': '#e9f5e6', 'edgecolor': '#2d8a3e'},
        'sn':   {**node_base, 'facecolor': '#e9f5e6', 'edgecolor': '#2d8a3e'},
        'pbc1': {**node_base, 'facecolor': '#fef3e6', 'edgecolor': '#d46b08'},
        'pbc2': {**node_base, 'facecolor': '#fef3e6', 'edgecolor': '#d46b08'},
        'pbc3': {**node_base, 'facecolor': '#fef3e6', 'edgecolor': '#d46b08'},
    }
    for node, (x, y) in pos.items():
        ax.text(x, y, labels[node], ha='center', va='center',
                fontsize=9, color='#1a1a1a', bbox=node_styles[node])

    STRUCT_PATHS = [
        ('ceek', 'pn'), ('ceea', 'pn'), ('eda', 'pn'),
        ('ceek', 'sn'), ('ceea', 'sn'), ('eda', 'sn'),
        ('pn', 'pbc1'), ('pn', 'pbc2'), ('pn', 'pbc3'),
        ('sn', 'pbc1'), ('sn', 'pbc2'), ('sn', 'pbc3'),
    ]

    for src, dst in STRUCT_PATHS:
        x0, y0 = pos[src]; x1, y1 = pos[dst]
        ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle='->', color='#666666', lw=1.0,
                                   connectionstyle='arc3,rad=0.0'), zorder=3)
        β_es = lookup_est(p_es, dst, src)
        mx, my = (x0 + x1) / 2, (y0 + y1) / 2
        if not np.isnan(β_es):
            ax.text(mx, my + 0.18, f'{β_es:.3f}', ha='center', va='bottom',
                    fontsize=8, color=C_ES, fontweight='bold', zorder=6)

    RESID_CORR = [
        ('pn', 'sn', 0.3),
        ('pbc1', 'pbc2', 0.2), ('pbc1', 'pbc3', 0.4), ('pbc2', 'pbc3', 0.2),
    ]
    for src, dst, rad in RESID_CORR:
        x0, y0 = pos[src]; x1, y1 = pos[dst]
        ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle='<->', color='#999999', lw=1.2,
                                   linestyle='dotted',
                                   connectionstyle=f'arc3,rad={rad}'), zorder=2)

    for x, lbl, col in [
        (1.3, 'Knowledge\n(BENCH K-activation)', '#2a78d6'),
        (4.5, 'Norms\n(Motivation)', '#2d8a3e'),
        (8.0, 'PBC\n(Consideration)', '#d46b08'),
    ]:
        ax.text(x, 7.7, lbl, ha='center', va='top', fontsize=8.5, color=col, fontstyle='italic')

    s_es = semopy.calc_stats(model_es).iloc[0]
    ax.text(0.02, 0.02,
            f'CFI={s_es["CFI"]:.3f},  RMSEA={s_es["RMSEA"]:.3f}  '
            f'(n={len(data_es):,})',
            transform=ax.transAxes, fontsize=8.5, va='bottom',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor='#cccccc', alpha=0.9))

    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    plt.tight_layout()
    plt.savefig('sem_path_diagram_m1b.png', dpi=180, bbox_inches='tight')
    plt.show()


draw_bench_sem_path_diagram(m1b_es)



## 12. Interpretation and Conclusions

### Model fit summary (M1b, primary model)
- **CFI = 0.993** — excellent fit
- **RMSEA = 0.049** — good fit (below 0.06 threshold)
- Fit achieved without adding direct knowledge → PBC paths (M1c worsens fit), confirming **full mediation through norms** as BENCH assumes

### Key findings

**1. Knowledge activates norms — BENCH core assumption supported**
All three knowledge composites (CEEK, CEEA, EDA) significantly predict personal norms (PN). EDA (energy decision awareness) has the strongest path to PN, suggesting that *decision-relevant* awareness drives moral obligation more than generic climate knowledge (CEEK). CEEA (awareness) has stronger links to SN than CEEK (knowledge), consistent with the idea that emotional engagement with climate issues promotes social norm perception.

**2. Knowledge → SN path is weaker — expected from the ABM structure**
In BENCH, social norms (SN) are primarily driven by *observing neighbours' renovation behaviour*, not by individual knowledge. The survey captures perceived social norms at a point in time, not the dynamic social learning process. A weaker knowledge → SN path is expected and not a model failure.

**3. Norms → PBC: all paths significant**
PN consistently has the stronger effect on investment PBC (pbc1) and switching PBC (pbc3). SN dominates for conservation PBC (pbc2) — possibly because conservation behaviours are more socially observable (lights off, thermostat settings) than structural investments.

**4. Full mediation confirmed**
Adding direct knowledge → PBC paths (M1c) worsens fit. This confirms the BENCH assumption that knowledge does not bypass norms — it must first activate motivation (PN/SN) before consideration (PBC) occurs.

### Limitations

- **Cross-sectional**: cannot establish causality; only structural consistency with the BENCH chain is tested
- **No intention measure**: BENCH's "consideration" step is inferred from PBC, not a separate intention item
- **SN dynamics not captured**: social norm updating from observing neighbours (the BENCH ABM mechanism) is absent in a one-shot survey
- **Binary behavioral outcomes** (Q801–Q807) treated as continuous composites — underestimates effect sizes for M3
- **Cross-loaded items** inflate reliability estimates — actual construct separation is lower than α suggests

### Implications for BENCH parameterisation

The differential effects of CEEK/CEEA/EDA on PN vs SN (which the BENCH ABM currently does not differentiate — it uses a single composite) could be used to:
1. Weight the three knowledge components differently in the guilt-activation threshold
2. Parameterise EDA specifically as the trigger for investment consideration
3. Use SN sensitivity to CEEA (not CEEK) to calibrate the social awareness updating mechanism


## 13. Publication-Quality Path Diagram (Li et al. Style)

Standardized path diagram matching Li et al. (2025) Figure 3 format:

- **Standardized regression weights** (model fitted on z-scored composites)
- **Arrow width** proportional to |β|; **solid** = significant (p < 0.05); **dashed** = non-significant
- **Dotted curved arrows** = residual correlations (not causal paths)
- **Stars**: *** p < 0.001, ** p < 0.01, * p < 0.05

In [ ]:
from scipy.stats import zscore as sp_zscore

# ── Standardized model fit (refit on z-scored composites) ───────────────────
_data_es = df_es[BEH_VARS].dropna()

def _std_fit(spec, data):
    dz = data.apply(sp_zscore, nan_policy='omit')
    m = semopy.Model(spec)
    m.fit(dz)
    p = m.inspect()
    reg = p[p['op'] == '~'][['lval', 'rval', 'Estimate', 'p-value']].copy()
    s = semopy.calc_stats(m).iloc[0]
    return reg, s

std_reg_es, std_fit_es = _std_fit(M1B_SPEC, _data_es)

def _lkp(reg, lhs, rhs):
    row = reg[(reg['lval'] == lhs) & (reg['rval'] == rhs)]
    if len(row) == 0:
        return np.nan, np.nan
    return float(row['Estimate'].iloc[0]), float(row['p-value'].iloc[0])

def _stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return ''

def _lbl(b, p):
    s = _stars(p)
    return f'{b:.3f}{s}' if s else '(ns)'

# ── Layout constants ─────────────────────────────────────────────────────────
NW, NH = 2.5, 1.1

NODE_POS = {
    'ceek': (1.5, 9.0),
    'ceea': (1.5, 5.5),
    'eda':  (1.5, 2.0),
    'pn':   (5.5, 7.5),
    'sn':   (5.5, 3.5),
    'pbc1': (10.0, 9.0),
    'pbc2': (10.0, 5.5),
    'pbc3': (10.0, 2.0),
}

NODE_LABELS = {
    'ceek': 'CEEK\nKnowledge',
    'ceea': 'CEEA\nAwareness',
    'eda':  'EDA\nDecision\nAwareness',
    'pn':   'Personal\nNorms (PN)',
    'sn':   'Social\nNorms (SN)',
    'pbc1': 'PBC1\nInvestment',
    'pbc2': 'PBC2\nConservation',
    'pbc3': 'PBC3\nSwitching',
}

NODE_FC = {n: '#d9eaf7' for n in ('ceek', 'ceea', 'eda')}
NODE_FC.update({'pn': '#d4efd5', 'sn': '#d4efd5'})
NODE_FC.update({'pbc1': '#fde8d0', 'pbc2': '#fde8d0', 'pbc3': '#fde8d0'})

NODE_EC = {n: '#1a4f99' for n in ('ceek', 'ceea', 'eda')}
NODE_EC.update({'pn': '#2d7a39', 'sn': '#2d7a39'})
NODE_EC.update({'pbc1': '#b05000', 'pbc2': '#b05000', 'pbc3': '#b05000'})


def _box_edge(cx, cy, tx, ty):
    dx, dy = tx - cx, ty - cy
    d = (dx**2 + dy**2) ** 0.5 or 1e-9
    ux, uy = dx / d, dy / d
    tw = (NW / 2 + 0.08) / (abs(ux) or 1e-9)
    th = (NH / 2 + 0.08) / (abs(uy) or 1e-9)
    t = min(tw, th)
    return cx + ux * t, cy + uy * t


PATH_SPECS = [
    ('ceek', 'pn',   0.10, 0.50,  0.30),
    ('ceea', 'pn',   0.00, 0.50, -0.35),
    ('eda',  'pn',  -0.22, 0.50,  0.35),
    ('ceek', 'sn',  -0.28, 0.50,  0.30),
    ('ceea', 'sn',   0.00, 0.50, -0.30),
    ('eda',  'sn',   0.12, 0.50, -0.30),
    ('pn',  'pbc1',  0.12, 0.50,  0.32),
    ('pn',  'pbc2',  0.00, 0.50, -0.32),
    ('pn',  'pbc3', -0.18, 0.50,  0.35),
    ('sn',  'pbc1',  0.22, 0.50,  0.32),
    ('sn',  'pbc2',  0.00, 0.50, -0.32),
    ('sn',  'pbc3', -0.12, 0.50, -0.32),
]

RESID_CORRS = [
    ('pn',   'sn',   0.45),
    ('pbc1', 'pbc2', 0.20),
    ('pbc2', 'pbc3', 0.20),
    ('pbc1', 'pbc3', 0.48),
]

# ── Single-panel figure ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 11))
fig.patch.set_facecolor('white')
ax.set_xlim(-0.5, 12.5)
ax.set_ylim(-1.2, 11.5)
ax.axis('off')
ax.set_facecolor('white')

for cx, lbl, ec in [
    (1.5,  'Knowledge\nActivation', '#1a4f99'),
    (5.5,  'Motivation\n(Norms)',   '#2d7a39'),
    (10.0, 'Consideration\n(PBC)', '#b05000'),
]:
    ax.text(cx, 11.1, lbl, ha='center', va='top', fontsize=10,
            color=ec, fontweight='bold', fontstyle='italic',
            multialignment='center')

for node, (cx, cy) in NODE_POS.items():
    ax.add_patch(mpatches.FancyBboxPatch(
        (cx - NW / 2, cy - NH / 2), NW, NH,
        boxstyle='round,pad=0.12',
        facecolor=NODE_FC[node], edgecolor=NODE_EC[node],
        linewidth=2.0, zorder=5,
    ))
    ax.text(cx, cy, NODE_LABELS[node], ha='center', va='center',
            fontsize=9.5, fontweight='bold', color='#111111',
            zorder=6, multialignment='center')

for src, dst, rad, t_pos, perp in PATH_SPECS:
    cx0, cy0 = NODE_POS[src]
    cx1, cy1 = NODE_POS[dst]
    xe0, ye0 = _box_edge(cx0, cy0, cx1, cy1)
    xe1, ye1 = _box_edge(cx1, cy1, cx0, cy0)

    b, p = _lkp(std_reg_es, dst, src)
    if np.isnan(b):
        continue
    is_sig = p < 0.05
    lw = max(1.5, 1.0 + 4.5 * abs(b)) if is_sig else 1.0
    ac = '#222222' if is_sig else '#cccccc'
    ls = 'solid' if is_sig else (0, (5, 4))

    ax.annotate('', xy=(xe1, ye1), xytext=(xe0, ye0),
                arrowprops=dict(
                    arrowstyle='->, head_width=0.22, head_length=0.20',
                    color=ac, lw=lw, linestyle=ls,
                    connectionstyle=f'arc3,rad={rad}',
                ), zorder=3)

    dx, dy = xe1 - xe0, ye1 - ye0
    dist = (dx**2 + dy**2) ** 0.5 or 1e-9
    lx = xe0 + t_pos * dx + perp * (-dy / dist)
    ly = ye0 + t_pos * dy + perp * (dx / dist)

    ax.text(lx, ly, _lbl(b, p), ha='center', va='center',
            fontsize=9 if is_sig else 8.5,
            color=C_ES if is_sig else '#aaaaaa',
            fontweight='bold' if is_sig else 'normal',
            zorder=7,
            bbox=dict(facecolor='white', edgecolor='none',
                      boxstyle='round,pad=0.15', alpha=0.9))

for rs, rd, rad_r in RESID_CORRS:
    x0, y0 = NODE_POS[rs]
    x1, y1 = NODE_POS[rd]
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(
                    arrowstyle='<->', color='#aaaaaa', lw=1.5,
                    linestyle=(0, (3, 2)),
                    connectionstyle=f'arc3,rad={rad_r}',
                ), zorder=2)

fit_txt = (
    f"CFI = {std_fit_es['CFI']:.3f}\n"
    f"RMSEA = {std_fit_es['RMSEA']:.3f}\n"
    f"χ²/df = {std_fit_es['chi2'] / std_fit_es['DoF']:.2f}   "
    f"df = {int(std_fit_es['DoF'])}\n"
    f"n = {len(_data_es):,}"
)
ax.text(0.01, 0.02, fit_txt, transform=ax.transAxes,
        fontsize=9, va='bottom', ha='left', color='#333333',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#f7f7f7',
                  edgecolor='#bbbbbb', alpha=0.95))

fig.text(
    0.5, -0.01,
    '*** p < 0.001    ** p < 0.01    * p < 0.05    (ns) not significant\n'
    'Standardized regression weights (z-scored composites).   '
    'Arrow width ∝ |β|.   Dashed = non-significant.   Dotted curved = residual correlation.',
    ha='center', va='top', fontsize=9, color='#555555',
)

ax.set_title(
    'BENCH Behavioural Framework — M1b Structural Path Model\n'
    'ES — Spain (Navarre, n = 755)   ·   Standardized regression weights',
    fontsize=13, fontweight='bold', pad=10,
)

plt.tight_layout(rect=[0, 0.04, 1, 1.0])
plt.savefig('sem_publication_figure.png', dpi=200,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: sem_publication_figure.png')

## 14. How to Interpret SEM Results — Plain-Language Guide

### What is SEM doing?

SEM tests whether a **specific causal structure** — e.g., "knowledge activates norms, which enable consideration" — is **statistically consistent with the observed data**. It runs multiple regressions simultaneously under the constraint that only certain causal links are allowed. The output tells you:

1. **Does the assumed structure fit the data?** → fit indices (CFI, RMSEA)
2. **How strong is each hypothesised link?** → path coefficients

---

### What does a standardized path coefficient mean?

A **standardized β** is the expected change in the **outcome** (in standard deviations) when the **predictor** increases by one standard deviation, holding all other predictors constant.

**Example: CEEA → SN = 0.329\*\*\***

> *"Households whose climate-energy awareness (CEEA) is one standard deviation above the mean are expected to perceive social norms 0.329 standard deviations stronger — after accounting for factual knowledge (CEEK) and energy decision awareness (EDA)."*

In Likert-scale terms:
- SD_CEEA ≈ 1.1 pts on the 1–7 scale → "one SD above mean" means awareness ≈ 5.3 vs mean ≈ 4.2
- SD_SN ≈ 1.2 pts → expected SN increase = 0.329 × 1.2 ≈ **0.4 Likert points**

**Contrast: CEEK → SN = 0.017 (ns)**

After controlling for awareness and decision relevance, knowing more factual facts about climate-economy has **no additional effect** on perceived social norms. Emotional engagement with the issue (CEEA) drives social norm perception — factual knowledge alone does not.

**Key paths:**
| Path | β | Meaning |
|------|---|---------|
| EDA → PN = 0.677\*\*\* | Largest | Decision-relevant awareness is the dominant driver of personal moral obligation |
| SN → PBC2 = 0.483\*\*\* | Strong | Social norms are the main driver of conservation feasibility perception |
| PN → PBC3 = 0.409\*\*\* | Strong | Personal moral norms strongly drive fuel-switching feasibility |
| CEEK → SN = 0.017 (ns) | Zero | Factual knowledge has no direct effect on social norms after controlling for awareness |

---

### What do significance stars mean?

| Stars | p-value threshold | Interpretation |
|-------|------------------|----------------|
| `***` | p < 0.001 | < 0.1% probability the true coefficient is zero |
| `**` | p < 0.01 | < 1% probability |
| `*` | p < 0.05 | < 5% probability (conventional threshold) |
| `(ns)` | p ≥ 0.05 | Cannot reject H₀: β = 0. Treat as "no evidence of an effect" |

Note: significance ≠ importance. A tiny β can be significant at large n. Always read the coefficient size alongside the stars.

---

### What do the fit indices mean?

| Index | Value | Threshold | Meaning |
|-------|-------|-----------|---------|
| **CFI** | 0.993 | ≥ 0.95 = good | Model reproduces 99.3% of the observed covariance structure |
| **RMSEA** | 0.049 | ≤ 0.06 = good | Average misfit per degree of freedom is small |
| **χ²/df** | 2.79 | ≤ 3.0 = acceptable | Omnibus test — always significant at n = 755; rely on CFI/RMSEA |

---

### What does "full mediation" mean for BENCH?

When we tested adding direct knowledge → PBC paths (model M1c), fit **worsened** (CFI: 0.993 → 0.942). This means:

> Knowledge reaches PBC **only through** norms — there is no direct shortcut.

```
CEEK ─┐
CEEA ─┼──→  PN  ──→  PBC1 / PBC2 / PBC3
EDA  ─┘     SN  ─/
```

The BENCH assumption is validated: knowledge activation must first convert into Motivation (PN + SN) before it can affect Consideration (PBC). An information campaign that raises awareness without activating moral concern or social norm perception is unlikely to move PBC — and therefore unlikely to drive renovation.